## Pandas – Basics 02

![](img/pandas.svg)

In this lesson, we're going to introduce some more fundamentals of [Pandas](https://pandas.pydata.org/pandas-docs/stable/getting_started/overview.html), a powerful Python library for working with tabular data like CSV files.

We will review skills learned from the last lesson and introduce how to:

* Broadly examine data
* Work with missing data
* Rename, drop, and add new columns
* Perform mathematical calculations
* Aggregate subsets of data
* Make a simple time series

The dataset that we're going to be working with in this lesson is taken from [The Trans-Atlantic Slave Trade Database](https://www.slavevoyages.org/voyage/database), part of the [*Slave Voyages* project](https://www.slavevoyages.org/). The larger database includes information about 35,000 slave-trading voyages from 1514-1866. The dataset we're working with here was filtered to include the 20,000 voyages that landed in the Americas. The data was filtered to also include the percentage of enslaved men, women, and children on the voyages.

---

### Set-up

As seen before:

In [ ]:
import pandas as pd
pd.options.display.max_rows = 100
slave_voyages_df = pd.read_csv('./datasets/data/Trans-Atlantic-Slave-Trade_Americas.csv', delimiter=",", encoding='utf-8')
slave_voyages_df.shape

---

### Dealing with missing data

#### .isna() / .notna()

Pandas has special ways of dealing with missing data. As you may have already noticed, blank rows in a CSV file show up as `NaN` in a Pandas DataFrame.

To filter and count the number of missing/not missing values in a dataset, we can use the special `.isna()` and `.notna()` methods on a DataFrame or Series object.

In [ ]:
slave_voyages_df['percent_women'].notna()

The `.isna()` and `.notna()` methods return True/False pairs for each row, which we can use to filter the DataFrame for any rows that have information in a given column. For example, we can filter the DataFrame for only rows that have information about the percentage of enslaved women aboard the voyage.

In [ ]:
slave_voyages_df[slave_voyages_df['percent_women'].notna()]

The data is now filtered to only include the 2,894 rows with information about how many women were aboard the voyage.

To explicitly count the number of blank rows, we can use the `.value_counts()` method.

In [ ]:
slave_voyages_df['percent_women'].isna().value_counts()

There are 17,874 that do not contain information about the number of enslaved women on the voyage (`isna` = True) and 2,894 rows that do contain this information (`isna` = False).

To quickly transform these numbers into percentages, we can set the `normalize=` parameter to True.

In [ ]:
slave_voyages_df['percent_women'].isna().value_counts(normalize=True)

About 14% of rows in this dataset have information about the number of enslaved women on the voyage while 86% do not.

#### .count()

Because the `.count()` method always excludes NaN values, we can also count the number of values in each column and divide by the total number of rows in each column (`len()`) to find the percentage of not blank data in every column.

In [ ]:
slave_voyages_df.count() / len(slave_voyages_df)

For example, 100% of the rows in the columns "year_of_arrival" contain information, while 2% of the rows in the column "resistance_label" contain information. The "resistance_label" indicates whether there is a record of the enslaved Africans aboard the voyage staging some form of resistance.

#### .fillna()

If we wanted, we could fill the `NaN` values in the DataFrame with a different value by using the `.fillna()` method.

In [ ]:
slave_voyages_df['percent_women'].fillna('no gender information recorded')

---

### Working with columns

#### Rename Columns

We can rename columns with the [`.rename()` method](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rename.html) and the `columns=` parameter. For example, we can rename the "flag" column "national_affiliation."

In [ ]:
slave_voyages_df.rename(columns={'flag': 'national_affiliation'})

Renaming the "flag" column as above will only momentarily change that column's name, however. If we display our DataFrame, we will see that the column name has *not* changed permamently.

In [ ]:
slave_voyages_df.head(1)

To save changes in the DataFrame, we need to reassign the DataFrame to the same variable.

In [ ]:
slave_voyages_df = slave_voyages_df.rename(columns={'flag': 'national_affiliation'})

In [ ]:
slave_voyages_df.head(1)

#### Drop Columns

We can remove a column from the DataFrame with the `.drop()` method and the column name.

In [ ]:
slave_voyages_dropped_df = slave_voyages_df.drop(columns="sources")
slave_voyages_dropped_df.columns

#### Add Columns

To add a column, we simply put a new column name in square brackets and set it equal to whatever we want the new column to be.

For example, if we wanted to create new columns for the total women and men aboard each voyage, we could set them equal to the product of the "total_disembarked" column * the "percent_women" / "percent_men" columns.

In [ ]:
slave_voyages_df['total_women'] = slave_voyages_df['total_embarked'] * slave_voyages_df['percent_women']

In [ ]:
slave_voyages_df['total_men'] = slave_voyages_df['total_embarked'] * slave_voyages_df['percent_men']

If we scroll all the way to the right side of the DataFrame, we can see that these columns have been added.

In [ ]:
slave_voyages_df.head(1)

#### Sort Columns

We can sort a DataFrame with the `.sort_values()` method, inside of which we include the parameter `by=` and indicate the name of the column we want to sort by (written in quotation marks).

For example, we can sort the DataFrame by the voyages that had the largest proportion of enslaved women aboard.

In [ ]:
slave_voyages_df.sort_values(by='percent_women', ascending=False)

By default, Pandas will sort in "ascending" order, from the smallest value to the largest value. If we want to sort the largest values first, we need to include another parameter `ascending=False`.

Because the DataFrame is truncated when it has more than 100 rows, we can use a Python list slice to view the top 30 (or any number less than 100) voyages with enslaved women aboard.

In [ ]:
slave_voyages_df.sort_values(by='percent_women', ascending=False)[:30]

If we want to sort a Series object, we don't need to use the `by=` paramter.

In [ ]:
slave_voyages_df['total_women'].sort_values(ascending=False)

#### Calculate Columns

We can do different calculations on columns with built-in Pandas functions. These calculations will ignore `NaN` values.

| Pandas calculations | Explanation                         |
|----------|-------------------------------------|
| `.count()`    | Number of observations    |
| `.sum()`      | Sum of values                       |
| `.mean()`     | Mean of values                      |
| `.median()`   | Median of values         |
| `.min()`      | Minimum                             |
| `.max()`      | Maximum                             |
| `.mode()`     | Mode                                |
| `.std()`      | Unbiased standard deviation         |



For example, to find the average proprotion of enslaved women aboard the voyages (for voyages that have this information), we can use the `.mean()` method.

In [ ]:
slave_voyages_df['percent_women'].mean()

There were on average 27% enslaved women aboard the voyages for voyages that recorded this information.

In [ ]:
slave_voyages_df['percent_women'].max()

The highest percentage of women aboard the slave voyages was 100%. We can use this calculation as a filter to identify the voyage(s) with this maximum value.

In [ ]:
slave_voyages_df[slave_voyages_df['percent_women'] == slave_voyages_df['percent_women'].max()]

According to the Trans-Atlantic Slave Trade Database, the 1819 voyage of the S José Diligente had 100% enslaved women aboard.

As demonstrated previously, we can also perform calculations with columns themselves.

In [ ]:
(slave_voyages_df['total_embarked'] * slave_voyages_df['percent_women']).max()

#### Groupby Columns

The Pandas function`.groupby()` allows us to group data and perform calculations on the groups.

For example, Jennifer Morgan writes about how some nations recorded more information about the gender of the enslaved people aboard their voyages than other nations did. To see the breakdown of gender information by nation, we can use a `.groupby()` function.

The first step to using groupby is to type the name of the DataFrame followed by `.groupby()` with the column we'd like to aggregate based on, such as "national_affiliation."

In [ ]:
slave_voyages_df.groupby('national_affiliation')

This action will created a [GroupBy object](https://pandas.pydata.org/pandas-docs/stable/user_guide/groupby.html). We can perform calculations on this grouped data, such as counting the number of non-blank values in each column for each nation.

In [ ]:
slave_voyages_df.groupby('national_affiliation').count()

For example, patterns emerge that suggest that English slave ship captains provided the most data related to the age or sex characteristics of the captives they transported and sold into slavery...The degree to which the practice of recording the sex of the passengers on board accords to national origin raises some interesting questions about the possible correlations between certain notational and national presumptions of accountability.

Jennifer Morgan, ["Accounting for 'The Most Excruciating Torment'"](https://read.dukeupress.edu/history-of-the-present/article-abstract/6/2/184/153282/Accounting-for-The-Most-Excruciating-Torment?redirectedFrom=PDF)

We can also isolate only the "percent_women" column.

In [ ]:
slave_voyages_df.groupby('national_affiliation').count()['percent_women']

In [ ]:
slave_voyages_df.groupby('national_affiliation')['percent_women'].count().sort_values(ascending=False)

If a line of code gets too long, you can create a line break with a backslash `\`

In [ ]:
slave_voyages_df.groupby('national_affiliation')['percent_women'].count()\
.sort_values(ascending=False).plot(kind='bar', title='Trans-Atlantic Slave Trade (Americas): \n Slave Voyages with Recorded Gender Information')

---

### Working with time series

#### Make Time Series with Groupby

To make a time series, we would typically want to convert our date column into datetime values rather than integers.

In [ ]:
slave_voyages_df['year_of_arrival'].dtype

Datetime values allow us to do special things that we can't do with regular integers and floats, such as extract just the year, month, week, day, or second from any date or aggregate based on any of the above.

However, we can also make some simple time series plots just by grouping by the year column and performing calculations on those year groupings, such as calculating the average percentage of enslaved women aboard the voyages over time.

In [ ]:
total_women_by_year = slave_voyages_df.groupby('year_of_arrival')['percent_women'].sum()

In [ ]:
total_women_by_year.plot()

In [ ]:
total_women_by_year.plot(kind='line', title="Trans-Atlantic Slave Trade (Americas):\nTotal Number of Enslaved Women on Voyages")

We can put different plots on the same axes by assigning one of the plots to the variable `ax`, short for axes, and then using `ax=ax` in the other plot to explicitly put it on the same axes.

In [ ]:
total_men_by_year = slave_voyages_df.groupby('year_of_arrival')['percent_men'].sum()

In [ ]:
ax = total_women_by_year.plot(kind='line', legend= True,
                              title="Trans-Atlantic Slave Trade (Americas):\nTotal Number of Enslaved Women on Voyages")
total_men_by_year.plot(ax=ax, legend=True)

We can change the labels in a legend by using the `label=` parameter.

In [ ]:
ax = total_women_by_year.plot(kind='line', label="Total Women", legend= True, title="Trans-Atlantic Slave Trade (Americas):\nTotal Number of Enslaved Women on Voyages")
total_men_by_year.plot(ax=ax, label="Total Men", legend=True)

Finally, we can also add in the total number of enslaved people who embarked on the voyages, offering a perspective of how much gender information we have about the voyages compared to the total number of voyages.

In [ ]:
total_embarked_by_year = slave_voyages_df.groupby('year_of_arrival')['total_embarked'].sum()

In [ ]:
ax = total_women_by_year.plot(kind='line', label="Total Women", legend= True, title="Trans-Atlantic Slave Trade (Americas):\nTotal Number of Enslaved Women on Voyages")
total_men_by_year.plot(ax=ax, label="Total Men", legend=True)
total_embarked_by_year.plot(ax=ax, label="Total Embarked", legend=True)

---

### Plots again

#### Save Plots

To save a plot as an image file or PDF file, we can again assign the plot to a variable called `ax`, short for axes.

Then we can use `ax.figure.savefig('FILE-NAME.png')` or `ax.figure.savefig('FILE-NAME.pdf')`.

In [ ]:
ax = total_women_by_year.plot(kind='line', label="Total Women", legend= True, title="Trans-Atlantic Slave Trade (Americas):\nTotal Number of Enslaved Women on Voyages")
total_men_by_year.plot(ax=ax, label="Total Men", legend=True)
total_embarked_by_year.plot(ax=ax, label="Total Embarked", legend=True)

ax.figure.savefig('Trans-Atlantic-Slave-Trade_Gender-Info.png')

#### Prevent Labels From Getting Cut Off

If labels are getting cut off in your image, you can explicitly import `matplotlib.pyplot` (the data viz Python library that Pandas `.plot()`s are built on) and use the `tight_layout()` function:

In [ ]:
import matplotlib.pyplot as plt

ax = total_women_by_year.plot(kind='line', label="Total Women", legend= True, title="Trans-Atlantic Slave Trade (Americas):\nTotal Number of Enslaved Women on Voyages")
total_men_by_year.plot(ax=ax, label="Total Men", legend=True)
total_embarked_by_year.plot(ax=ax, label="Total Embarked", legend=True)

plt.tight_layout()
ax.figure.savefig('Trans-Atlantic-Slave-Trade_Gender-Info.png')